In [ ]:
!curl "https://raw.githubusercontent.com/philipturner/swift-colab/release/latest/install_swift.sh" --output "install_swift.sh"
!bash "install_swift.sh" "5.6.2" #// Replace '5.6.2' with newest Swift version.
#// After this cell finishes, go to 'Runtime > Restart runtime'.

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  7845  100  7845    0     0  33101      0 --:--:-- --:--:-- --:--:-- 33101
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  489M  100  489M    0     0  26.5M      0  0:00:18  0:00:18 --:--:-- 26.3M
Cloning into 'swift-colab'...
remote: Enumerating objects: 120, done.
remote: Counting objects: 100% (120/120), done.
remote: Compressing objects: 100% (112/112), done.
remote: Total 120 (delta 7), reused 91 (delta 7), pack-reused 0
Receiving objects: 100% (120/120), 294.87 KiB | 1.54 MiB/s, done.
Resolving deltas: 100% (7/7), done.
Compiling Swift LLDB bindings
Compiling JupyterKernel
Adding symbolic link to JupyterKernel binary
Registering Swift Jupyter kernel
=== -------------------------------------------------------

In [ ]:
// Assume 5x5x5 cube of ions, but four edges are cut off to approximate a
// sphere. If the electric potential energy between two adjacent atoms is -1,
// what is the entire system's electric potential energy?
import Foundation

// Sheet of integer coordinates in the lattice.
func generateSheet(z: Int) -> [SIMD3<Int>] {
  var output: [SIMD3<Int>] = []
  for x in -2...2 {
    for y in -2...2 {
      if abs(x) == 2 && abs(y) == 2 {
        continue
      }

      output.append([x, y, z])
    }
  }

  precondition(output.count == 21)
  return output
}

let lattice = Range<Int>(-2...2).flatMap { z in
  return generateSheet(z: z)
}

In [ ]:
// Create a 10x10x10 cylinder of ions, where the cross section has 80 atoms.
// This is accomplished by etching 5 atoms off each corner of a 10x10 square.
// 0 0 0 X X X
// 0 X X X X X <-- The 5 zeroes here are the arrangement we need.
// 0 X X X X X
// X X X X X X
import Foundation

// Sheet of integer coordinates in the lattice.
func generateSheet(z: Int) -> [SIMD3<Int>] {
  var output: [SIMD3<Int>] = []
  for x in 0...9 {
    for y in 0...9 {
      // Corners are 0...2 and 7...9 inclusive.
      func exceedsUpperY() -> Bool {
        if y == 0 || y == 9 {
          return true
        } else {
          return false
        }
      }
      func exceedsLowerY() -> Bool {
        if y <= 2 || y >= 7 {
          return true
        } else {
          return false
        }
      }

      if x == 0 && exceedsLowerY() { continue }
      if x <= 2 && exceedsUpperY() { continue }
      if x >= 7 && exceedsUpperY() { continue }
      if x == 9 && exceedsLowerY() { continue }

      output.append([x, y, z])
    }
  }

  return output
}

// Modified `printSheet` from below, but uses different bounds.
func printSheet(_ sheet: [SIMD3<Int>]) {
  let sampleRow = [Bool](repeating: false, count: 10)
  var rows: [[Bool]] = Array(repeating: sampleRow, count: 10)
  for element in sheet {
    let columnID = element.x 
    let rowID = element.y
    precondition(rows[rowID][columnID] == false)
    rows[rowID][columnID] = true
  }

  for row in rows {
    var string = ""
    for element in row {
      if element == true {
        string += "X"
      } else {
        string += " "
      }
    }
    print(string)
  }
}

let sheet = generateSheet(z: 0)
printSheet(sheet)

let lattice = Range<Int>(0...9).flatMap { z in
  return generateSheet(z: z)
}

   XXXX   
 XXXXXXXX 
 XXXXXXXX 
XXXXXXXXXX
XXXXXXXXXX
XXXXXXXXXX
XXXXXXXXXX
 XXXXXXXX 
 XXXXXXXX 
   XXXX   


In [ ]:
// Retry conditions above, but remove every other ion. The central ion stays
// intact. Also create a way to visualize each layer, and ensure it's correct.
import Foundation

// Sheet of integer coordinates in the lattice.
func generateSheet(z: Int) -> [SIMD3<Int>] {
  var output: [SIMD3<Int>] = []
  for x in -2...2 {
    for y in -2...2 {
      if abs(x) == 2 && abs(y) == 2 {
        continue
      }
      if (x + y + z) % 2 != 0 {
        continue
      }
      output.append([x, y, z])
    }
  }
  return output
}

func printSheet(_ sheet: [SIMD3<Int>]) {
  let sampleRow = [Bool](repeating: false, count: 5)
  var rows: [[Bool]] = Array(repeating: sampleRow, count: 5)
  for element in sheet {
    let columnID = element.x + 2
    let rowID = element.y + 2
    precondition(rows[rowID][columnID] == false)
    rows[rowID][columnID] = true
  }

  for row in rows {
    var string = ""
    for element in row {
      if element == true {
        string += "X"
      } else {
        string += " "
      }
    }
    print(string)
  }
}

let lattice = Range<Int>(-2...2).flatMap { z -> [SIMD3<Int>] in
  let sheet = generateSheet(z: z)
  print("z-coordinate: \(z)")
  printSheet(sheet)
  print()
  return sheet
}
print("Number of ions: \(lattice.count)")

z-coordinate: -2
  X  
 X X 
X X X
 X X 
  X  

z-coordinate: -1
 X X 
X X X
 X X 
X X X
 X X 

z-coordinate: 0
  X  
 X X 
X X X
 X X 
  X  

z-coordinate: 1
 X X 
X X X
 X X 
X X X
 X X 

z-coordinate: 2
  X  
 X X 
X X X
 X X 
  X  

Number of ions: 51


In [ ]:
func getDistance(_ a: SIMD3<Int>, _ b: SIMD3<Int>) -> Float {
  precondition(a != b, "Exclude interactions of a point with itself.")
  let delta = SIMD3<Float>(b &- a)
  let len = (delta * delta).sum().squareRoot()
  return len
}

func showEnergy(lattice: [SIMD3<Int>]) {
  var sum: Float = 0

  // No two points have the same coordinate
  for point in lattice {
    for other in lattice {
      if point == other {
        continue
      }
      let distance = getDistance(point, other)
      let potential = -1 / distance
      sum += potential
    }
  }

  print("Total electric potential energy: \(sum)")
  print("Energy per ion: \(sum / Float(lattice.count))")
}

showEnergy(lattice: lattice)

Total electric potential energy: -129158.266
Energy per ion: -161.44783
